In [ ]:
!pip install phetk --upgrade

In [ ]:
from phetk.cohort import Cohort
from phetk.phecode import Phecode
from phetk.phewas import PheWAS
from phetk.plot import Plot


In [ ]:
from importlib.metadata import version

print("PheTK version:", version("phetk"))

In [ ]:
%pip install duckdb==1.3.2

In [ ]:
import duckdb

print("DuckDB version:", duckdb.__version__)
print("Supports .pl():", hasattr(duckdb.connect(), "pl"))

In [ ]:
%%time

phecode = Phecode(platform="aou")

phecode.count_phecode(
    phecode_version="X",
    output_file_path="aou_phecode_counts.tsv"
)

In [ ]:
import os

os.environ["WORKSPACE_REFERENCED_BUCKET"] = os.environ["WORKSPACE_REFERENCED_BUCKET_2"]

print("Reference bucket:", os.environ["WORKSPACE_REFERENCED_BUCKET"])

In [ ]:
%%time

cohort = Cohort(platform="aou", aou_db_version=9)

cohort.by_genotype(
    chromosome_number=1,
    genomic_position=152313385,
    ref_allele="G",
    alt_allele="A",
    gt_dict={0: "0/0", 1: ["0/1", "1/1"]},
    reference_genome="GRCh38",
    output_file_path="rs61816761_cohort.tsv"
)

In [ ]:
%%time

cohort.add_covariates(
    cohort_file_path="rs61816761_cohort.tsv",
    age_at_last_ehr_event=True,
    sex_at_birth=True,
    first_n_pcs=5,
    drop_nulls=True,
    output_file_path="rs61816761_cohort_with_covariates.tsv"
)

In [ ]:
%%time

phewas = PheWAS(
    phecode_version="X",
    phecode_count_file_path="aou_phecode_counts.tsv",
    cohort_file_path="rs61816761_cohort_with_covariates.tsv",
    covariate_cols=[
        "age_at_last_ehr_event",
        "sex_at_birth",
        "pc1",
        "pc2",
        "pc3",
        "pc4",
        "pc5"
    ],
    independent_variable_of_interest="genotype",
    sex_at_birth_col="sex_at_birth",
    min_cases=50,
    min_phecode_count=2,
    method="logit",
    output_file_path="rs61816761_phewas_results.tsv"
)

phewas.run()

In [ ]:
import polars as pl

results = pl.read_csv(
    "rs61816761_phewas_results.tsv",
    separator="\t"
)

print("Columns:", results.columns)
print("\nFirst 5 rows:")
print(results.head())

In [ ]:
top10 = (
    results
    .filter(pl.col("converged") == True)
    .sort("p_value")
    .select([
        "phecode",
        "phecode_string",
        "cases",
        "controls",
        "p_value",
        "odds_ratio",
        "neg_log_p_value"
    ])
    .head(10)
)

top10.write_csv(
    "rs61816761_top10_associations.tsv",
    separator="\t"
)

print("Top 10 associations saved successfully!")
print(top10)

In [ ]:
import numpy as np

In [ ]:
plot = Plot(
    phewas_result_file_path="rs61816761_phewas_results.tsv",
    phecode_version="X",
    bonferroni=-np.log10(0.05 / 2632)
)

plot.manhattan(
    label_values="p_value",
    label_count=10,
    label_text_column="phecode_string",
    title="FLG rs61816761 PheWAS Manhattan Plot",
    dpi=300,
    save_plot=True,
    output_file_path="rs61816761_manhattan_corrected.png"
)
